#### # 1. Multi-Table Ingestion & Alias Definition

In [0]:
import pyspark.sql.functions as F

# Ingest the clean Silver sales detail records and Gold dimensions using explicit aliases
sales_details = spark.table("workspace.silver.crm_sales_details").alias("sd")
dim_products  = spark.table("workspace.gold.dim_products").alias("pr")
dim_customers = spark.table("workspace.gold.dim_customers").alias("cu")

#### # 2. Execute Relational Left Joins to Map Dimension Keys

In [0]:
# Join the transaction records to product and customer dimensions based on your verified schema columns
joined_df = (
    sales_details
    .join(dim_products, F.col("sd.product_number") == F.col("pr.product_number"), "left")
    .join(dim_customers, F.col("sd.customer_id") == F.col("cu.customer_id"), "left")
)

#### # 3. Dimensional Schema Selection, Casting, and Storage

In [0]:
# Single-Pass Operation: Extract specific keys, enforce correct data types, and format final schema
final_df = joined_df.select(
    F.col("sd.order_number").cast("string").alias("order_number"),
    F.col("pr.product_key").cast("integer").alias("product_key"),
    F.col("cu.customer_key").cast("integer").alias("customer_key"),
    F.col("sd.order_date").cast("date").alias("order_date"),
    F.col("sd.shipping_date").cast("date").alias("shipping_date"),
    F.col("sd.due_date").cast("date").alias("due_date"),
    F.col("sd.sales_amount").cast("double").alias("sales_amount"),
    F.col("sd.quantity").cast("integer").alias("quantity"),
    F.col("sd.price").cast("double").alias("price")
)

# Commit the finalized dataframe directly as a production-grade Gold Delta table
final_df.write \
    .mode("overwrite") \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.fact_sales")

# Display a clean data preview to inspect column alignment and verify row integrity
# Check the full written Gold table without any limits applied first
spark.table("workspace.gold.fact_sales") \
     .filter(F.col("quantity") == 2) \
     .display()